In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: GEO02 - Business Travel
   
   UPDATED LOGIC:
   1. MASTER AU ANCHOR: Uses a FULL OUTER JOIN between the Master AU List 
      (fy25_list_of_unit) and the Mapping Bootstrap. This guarantees EVERY Master AU 
      is listed (even with 0 data), and catches rogue AUs that have mapped 
      data but are missing from the Master List.
   2. DYNAMIC APPLICABILITY: Automatically sets 'Not Applicable' and provides a 
      rationale if the AU lacks mapped cost centers or is missing from the Master list.
   3. FLAGS: Adds specific columns to show if the AU is in the Master List and 
      if it actually registered any trips.
   4. TRAVEL LOGIC: Unions AMEX/CWT, filters by High Risk ABAC countries, and 
      sums the trips.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab every single legitimate AU from the Master List
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
),

Mapped_AUs AS (
    -- 2. Grab every AU that currently has cost centers mapped to it
    SELECT DISTINCT AU_ID 
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
),

All_Base_AUs AS (
    -- 3. Combine Master List and Mapping to create the ultimate base table
    SELECT 
        COALESCE(mast.BusinessID, map.AU_ID) AS Base_AU_ID,
        mast.AU_Name,
        CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_Master_List,
        CASE WHEN map.AU_ID IS NOT NULL THEN 'Yes' ELSE 'No' END AS Has_Mapping
    FROM Master_AUs mast
    FULL OUTER JOIN Mapped_AUs map 
        ON mast.BusinessID = map.AU_ID
),

High_Risk_Countries AS (
    -- 4. Grab high-risk jurisdictions
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Combined_Travel_Data AS (
    -- 5. Union the AMEX and CWT files 
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        TRY_CAST(TRIM(Numberoftrips) AS INT) AS Trip_Count
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
    
    UNION ALL
    
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        1 AS Trip_Count
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
),

High_Risk_Trips AS (
    -- 6. Filter travel data to strictly keep high-risk destinations
    SELECT 
        t.Cleaned_CC,
        t.Trip_Count
    FROM Combined_Travel_Data t
    INNER JOIN High_Risk_Countries h
        ON t.DestinationCountry = h.CountryName
    WHERE t.Cleaned_CC IS NOT NULL
),

Trips_By_AU AS (
    -- 7. Map the high-risk trips to the AU_ID and sum up the trip count
    SELECT 
        m.AU_ID,
        SUM(h.Trip_Count) AS Total_Trips
    FROM vw_cost_center_mapping_bootstrap m
    INNER JOIN High_Risk_Trips h
        ON CAST(m.Cost_Center_ID AS INT) = h.Cleaned_CC
    GROUP BY m.AU_ID
)

-- 8. Final Template: Attach data and dynamic logic to the All_Base_AUs table
SELECT 
    a.Base_AU_ID AS BusinessID,                          
    a.AU_Name,                             
    'GEO02' AS QuestionID,               
    
    -- Dynamic Applicability Logic
    CASE 
        WHEN a.In_Master_List = 'No' THEN 'Not Applicable'
        WHEN a.Has_Mapping = 'No' THEN 'Not Applicable'
        ELSE 'Applicable' 
    END AS Applicability,       
    
    -- Dynamic Rationale Logic
    CASE 
        WHEN a.In_Master_List = 'No' THEN 'AU found in Mapping table but missing from Master List'
        WHEN a.Has_Mapping = 'No' THEN 'No Cost Centers mapped to this AU'
        ELSE '' 
    END AS ApplicabilityRationale,        
    
    -- Output total sum, defaulting to 0 for AUs with no trips
    COALESCE(CAST(t.Total_Trips AS STRING), '0') AS Response,
    
    -- Requested Validation Columns appended to the end
    a.In_Master_List,
    CASE WHEN t.Total_Trips > 0 THEN 'Yes' ELSE 'No' END AS Has_Data
    
FROM All_Base_AUs a
LEFT JOIN Trips_By_AU t 
    ON a.Base_AU_ID = t.AU_ID
ORDER BY a.Base_AU_ID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: GEO02 - Business Travel Drill-Down
=================================================================================== */

WITH Master_AUs AS (
    SELECT TRIM(CAST(business_ID AS STRING)) AS BusinessID
    FROM hive_metastore.ra_adido_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Combined_Travel_Data AS (
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        TRY_CAST(TRIM(Numberoftrips) AS INT) AS Trip_Count,
        'AMEX' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
    UNION ALL
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        1 AS Trip_Count,
        'CWT' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
)

SELECT DISTINCT
    m.AU_ID AS BusinessID,
    m.AU_Name,
    CASE WHEN mast.BusinessID IS NOT NULL THEN 'Yes' ELSE 'No' END AS In_Master_List,
    m.Cost_Center_ID AS Mapped_Cost_Center,
    t.Source_System,
    t.DestinationCountry AS High_Risk_Destination,
    t.Trip_Count
    
FROM vw_cost_center_mapping_bootstrap m

INNER JOIN Combined_Travel_Data t
    ON CAST(m.Cost_Center_ID AS INT) = t.Cleaned_CC
    
INNER JOIN High_Risk_Countries h
    ON t.DestinationCountry = h.CountryName
    
LEFT JOIN Master_AUs mast 
    ON m.AU_ID = mast.BusinessID
    
ORDER BY BusinessID;